# UR3e Basics

Quick reference for controlling the UR3e via `airo_robots`. Robot must be in **Remote Control** mode on the pendant. Keep the emergency button close whenever the robot moves.

In [4]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

from airo_robots.manipulators.hardware.ur_rtde import URrtde
from airo_robots.grippers.hardware.robotiq_2f85_urcap import Robotiq2F85

ip_address = "10.42.0.162"  # fill in the robot IP

robot = URrtde(ip_address, URrtde.UR3E_CONFIG)
#gripper = Robotiq2F85(ip_address)  # only if a gripper is attached

## Reading robot state

In [5]:
q = robot.get_joint_configuration()  # 6 joint angles in rad
X_B_TCP = robot.get_tcp_pose()  # 4x4 homogeneous matrix, TCP in base frame
q, X_B_TCP

(array([ 0.326, -0.125, -1.313, -0.189,  1.547, -0.526]),
 array([[-0.198,  0.978,  0.06 , -0.277],
        [-0.98 , -0.199,  0.002, -0.238],
        [ 0.014, -0.058,  0.998,  0.722],
        [ 0.   ,  0.   ,  0.   ,  1.   ]]))

## Joint-space movement (MoveJ)

Movement calls are asynchronous, `.wait()` blocks until the move finishes.

In [6]:
q_home = [0, -np.pi / 2, 0, -np.pi / 2, 0, 0]  # define a comfortable home configuration
robot.move_to_joint_configuration(q_home).wait()

<ACTION_STATUS_ENUM.SUCCEEDED: 2>

In [7]:
robot.move_to_joint_configuration(q).wait()

<ACTION_STATUS_ENUM.SUCCEEDED: 2>

## Task-space movement (MoveJ/MoveL to a pose)

`move_to_tcp_pose` ~ MoveJ, `move_linear_to_tcp_pose` ~ MoveL (straight line in Cartesian space).

In [8]:
X_B_TCP = robot.get_tcp_pose()

X_target = X_B_TCP.copy()
X_target[0, 3] += 0.10  # move 10 cm in +x of the base frame

robot.move_to_tcp_pose(X_target).wait()
robot.move_linear_to_tcp_pose(X_B_TCP).wait()  # move back, straight line

<ACTION_STATUS_ENUM.SUCCEEDED: 2>

## Building an orientation

A rotation matrix is 3 orthonormal column vectors [X, Y, Z] expressing the TCP axes in the base frame.

In [ ]:
# example: top-down orientation (TCP z-axis pointing down into the table)
X = np.array([1, 0, 0])
Z = np.array([0, 0, -1])
Y = np.cross(Z, X)
R_B_TCP = np.column_stack([X, Y, Z])

p_B_TCP = np.array([0.3, -0.1, 0.2])  # target position

X_B_TCP_target = np.identity(4)
X_B_TCP_target[:3, :3] = R_B_TCP
X_B_TCP_target[:3, 3] = p_B_TCP

robot.move_to_tcp_pose(X_B_TCP_target).wait()

## Freedrive (teach mode)

Use this from Python instead of the pendant, otherwise you have to restart the notebook when switching back from local mode.

In [ ]:
robot.rtde_control.teachMode()
# move the arm by hand, then read out its configuration
robot.get_joint_configuration()

In [ ]:
robot.rtde_control.endTeachMode()

## Gripper (Robotiq 2F-85)

In [ ]:
gripper.open().wait()
gripper.close().wait()
gripper.move(0.03, speed=0.1, force=10).wait()  # width in m, speed in m/s, force in N

## Force mode

Lets the robot comply with external force instead of holding a fixed position, e.g. useful for pushing down onto a surface until contact.

In [ ]:
import time

task_frame = [0, 0, 0, 0, 0, 0]  # base frame
compliant_axes = [1, 1, 1, 0, 0, 0]  # position compliant, orientation locked
force_type = 2
limits = [0.2, 0.2, 0.2, 1, 1, 1]  # max speed 20 cm/s
desired_force = [0, 0, -5, 0, 0, 0]  # push down with 5 N

robot.rtde_control.zeroFtSensor()  # calibrate the force measurement first
robot.rtde_control.forceMode(task_frame, compliant_axes, desired_force, force_type, limits)

for _ in range(10):
    print(np.array(robot.rtde_receive.getActualTCPForce())[:3])
    time.sleep(1)

robot.rtde_control.forceModeStop()

## Before shutting down

Put the robot in a safe position (away from objects, no links hanging outside the table) using freedrive.

In [ ]:
robot.rtde_control.teachMode()
# move to a safe pose by hand
robot.rtde_control.endTeachMode()